# CS2 (국내반) --- Assignment #2: One-to-Many Stable Matching
### Problem Set (assignment)

*9-10주차 수업의 stable matching(일대일)을 일대다(one-to-many) 매칭으로 일반화한 과제입니다.*

#### 문제 상황

다음과 같은 (실제에 매우 가까운) 가상의 상황을 고려하자.

- (i) n명의 교사 T1, T2, ..., Tn과 m명의 학생 S1, S2, ..., Sm 간에 연구활동 지도를 위한
  일대다(one-to-many) 매칭을 하려고 한다.
- (ii) 각 학생 Sj는 **정확히 한 명**의 교사에게 지도받아야 한다.
- (iii) 각 교사 Ti는 **정확히 ℓi (> 0)명**의 학생들을 지도해야 한다.
  - ℓ1 + ℓ2 + ... + ℓn = m을 만족하도록 주어져 있다.
- (iv) 각 교사 Ti는 학생들에 대한 preference list를 가진다.
- (v) 각 학생 Sj는 교사들에 대한 preference list를 가진다.
- (vi) 조건 (ii)-(iii)을 만족하는 matching M을 구성하되, 다음 조건들을 모두 만족하는
  (Ti, Ti', Sj, Sj')가 존재하지 않아야 한다 (**stability**):
  - i ≠ i'이고 j ≠ j'이다.
  - 학생 Sj'는 교사 Ti에게 매칭되어 있다.
  - 학생 Sj는 교사 Ti'에게 매칭되어 있다.
  - 교사 Ti는 학생 Sj'보다 학생 Sj를 선호한다.
  - 학생 Sj는 교사 Ti'보다 교사 Ti를 선호한다.

이런 (Ti, Sj)를 **unstable pair**라 부른다.

#### 알고리즘 (Propose-Reject의 일대다 버전)

```
각 교사/학생을 매칭하지 않은 상태로 초기화

while 어떤 교사 Ti에게 배정되어 있는 학생 수가 ℓi 미만이고
      Ti에게 배정을 아직 시도하지 않은 학생이 있음:
    이런 교사 Ti를 임의로 선택
    Sj = Ti에 배정을 시도하지 않은 학생 중 Ti가 가장 선호하는 학생

    # 학생 Sj를 교사 Ti에게 배정을 시도하되, Sj의 눈치작전 상황을 흉내내자
    if 학생 Sj가 아직 아무에게도 배정되지 않음:
        학생 Sj를 교사 Ti에 (잠시) 배정
    else:
        if 학생 Sj가 교사 Ti를 현재 배정되어 있는 교사 Tk보다 선호:
            학생 Sj와 교사 Tk 간의 매칭을 취소
            학생 Sj를 교사 Ti에게 (잠시) 배정
        else:
            pass    # 매칭에 변경을 가할 것이 없음
```

### Problem 1

(Correctness) 제시한 알고리즘이 조건 (ii), (iii), (vi)를 모두 만족하는 matching을 찾아주는
정확한 알고리즘임을 보이기 위해 다음을 차례대로 증명하라.
(10주차 수업자료의 증명을 완벽히 이해한 후 이를 흉내내면 된다.)

- (a) 위 알고리즘에서 while 루프는 최대 n·m회 반복한 후 종료한다. (**Termination**)
- (b) 위 알고리즘이 생성하는 matching은 조건 (ii)와 (iii)을 만족한다. (**Perfection**)
- (c) 위 알고리즘이 생성하는 matching은 조건 (vi)을 만족한다. (**Stability**)

*아래 셀을 더블클릭해서 증명을 작성하세요.*

**(a) Termination**

(여기에 작성)

**(b) Perfection**

(여기에 작성)

**(c) Stability**

(여기에 작성)

#### 먼저 실행: `Teacher` class와 `Student` class

아래 셀은 **수정하지 않고 맨 먼저 실행**합니다.

- 수업 때는 남/여를 모두 `Person` class로 표현했지만, 이 문제에서는 `Teacher` class와 `Student` class로 분리
- `Teacher`의 `self.quota`는 조건 (iii)의 ℓi, `self.apprentice`는 현재 배정된 학생들의 list
- `quotaAvailable()`: 학생을 추가로 받을 자리가 남았는지 여부 (boolean)
- `nextStudent()`: 수업 때 다룬 `Person` class의 `womanToPropose`와 동일
- `Student`의 `engage(t)`와 `breakUp()`으로 교사-학생 관계를 변경
  - `engage`는 학생에게 지도교사가 없고 교사에게 자리가 남았을 때(`quotaAvailable()`)만 사용

In [ ]:
class Teacher(object):
    def __init__(self, quota):
        self.prefList = None  # preference list (of Student objects)
        self.quota = quota    # maximum possible number of students
        self.apprentice = []  # list of students to supervise
        self.numProposal = 0

    def quotaAvailable(self):
        return len(self.apprentice) < self.quota

    def nextStudent(self):
        assert self.numProposal < len(self.prefList)
        s = self.prefList[self.numProposal]
        self.numProposal += 1
        return s

####################################################################
class Student(object):
    def __init__(self, id):
        self.prefList = None  # preference list (of Teacher objects)
        self.advisor = None   # teacher who will supervise self's research
        self.id = id          # for printing/debugging

    def isFree(self):
        return self.advisor == None

    def prefer(self, p):  # whether or not self prefer p to current advisor
        assert type(p) == Teacher
        assert not self.isFree()
        i = self.prefList.index(self.advisor)
        j = self.prefList.index(p)
        return j < i

    def engage(self, p):
        assert type(p) == Teacher
        assert self.isFree()
        assert self not in p.apprentice and p.quotaAvailable()
        self.advisor = p
        p.apprentice.append(self)

    def breakUp(self):
        assert not self.isFree()
        assert self in self.advisor.apprentice
        self.advisor.apprentice.remove(self)
        self.advisor = None

#### 확인용 함수 (실행만 하면 됨)

`test()`는 50개의 입력에 대해 `findMatch`가 만든 matching이 perfect하고 stable한지 검사합니다.
모두 통과하면 마지막에 `Correct for all 50 inputs`가 출력됩니다.

In [ ]:
import random

def _tPrefer(t, p, q):  # whether teacher t prefers student p to student q
    return t.prefList.index(p) < t.prefList.index(q)

def isPerfectMatch(teachers, students):
    matched = 0
    for s in students:
        if type(s.advisor) == Teacher:
            matched += 1
            if s not in s.advisor.apprentice:
                return False
    if matched != len(students):
        return False
    total = 0
    for t in teachers:
        l = len(t.apprentice)
        if l != t.quota or len(set(t.apprentice)) != l:
            return False
        total += l
    return total == matched

def isStableMatch(teachers, students):
    for p in students:          # p = S_j  (matched to v = T_i')
        for q in students:      # q = S_j' (matched to u = T_i)
            if p is q:
                continue
            u, v = q.advisor, p.advisor
            if u != v and _tPrefer(u, p, q) and p.prefer(u):
                return False    # (u, p) is an unstable pair
    return True

def genData(seed):
    n = 2 + seed // 2
    quota = 10 - n // 5
    m = n * quota
    teachers = [Teacher(quota) for i in range(n)]
    students = [Student(j) for j in range(m)]
    rng = random.Random(seed)
    for t in teachers:
        t.prefList = students[:]
        rng.shuffle(t.prefList)
    for s in students:
        s.prefList = teachers[:]
        rng.shuffle(s.prefList)
    return teachers, students

def printMatch(teachers):
    for i, t in enumerate(teachers):
        print(i, [s.id for s in t.apprentice])

def test(T=50):
    correct = True
    for h in range(T):
        teachers, students = genData(h)
        findMatch(teachers, students)
        perfect = isPerfectMatch(teachers, students)
        stable = perfect and isStableMatch(teachers, students)
        if not (perfect and stable):
            print("Incorrect output for %d-th input" % (h + 1), end=" ")
            print(": not even perfect" if not perfect else ": perfect but not stable")
            correct = False
        else:
            print("Correct output for %d-th input" % (h + 1))
    if correct:
        print("Correct for all %d inputs" % T)

### Problem 2

(Implementation) 제시한 알고리즘을 Python 프로그램으로 구현하라.

Write a function `findMatch`:
- input parameter: `Teacher` object들의 list `teachers`, `Student` object들의 list `students`
- return value: 없음 (각 object의 `apprentice`/`advisor`가 최종 matching을 나타내도록 변경)
- `Teacher` class와 `Student` class의 함수들은 **변경하지 말고** 제공된 그대로 사용
- class의 state 변수들을 바로 읽고 쓰는 것은 필요하지도 않고 권장하지도 않음
- *수업 때 다룬 `findMatch`를 흉내내면 된다.*

In [ ]:
def findMatch(teachers, students):
    n = len(teachers)
    m = len(students)

    # ADD ADDITIONAL CODE HERE!!

    pass

# ===== 확인용 (수정하지 말고 그대로 실행) =====
test()

### Problem 3 (0 points)

답을 제출해야 하는 문제는 아닌데 재미삼아 고민해보는 것을 추천한다.

- 알고리즘 상에서는 학생이 주도적으로 눈치작전을 사용하는 것 같은데, 실제로 생성된 매칭은
  교사, 학생 중 어느 편이 유리할까?
  - 10주차 수업 자료의 man-optimality, woman-pessimality 참조
- 그렇다면 어떤 식으로 알고리즘을 변경하면 학생 편이 유리해질 수 있을까?
- 어느 한쪽만 극단적으로 유리하지 않고 양쪽 모두에게 fair한 stable matching을 찾으려면
  어떻게 해야 할까?

*(생각을 적어보세요)*